In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 6 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Week 11 improved strongly compared with Week 10 and landed
# close to the incumbent.
#
# Recent points moved mainly toward lower x3 / x4 and did not
# beat the incumbent, so Week 12 tests whether the local optimum
# lies closer to the incumbent or slightly beyond it.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------

week11_pred_mean = -0.23128728702383516
week11_pred_std = 0.03895291141835071

week11_actual = -0.20392262491163674

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week11_pred_mean)
print("Predicted std :", week11_pred_std)
print("Actual        :", week11_actual)

print("\nPrediction error:")
print(week11_error)

print("\nError / predicted std:")
print(week11_z_error)


# ------------------------------------------------------------
# 3. Recent observations
# ------------------------------------------------------------

week10_x = np.array([
    0.445661,
    0.406946,
    0.588674,
    0.702373,
    0.121625
])

week10_y = -0.2629443056177093

week11_x = np.array([
    0.444632,
    0.410696,
    0.586583,
    0.711960,
    0.118032
])

week11_y = -0.20392262491163674

print("\n================================")
print("RECENT LOCAL POINTS")
print("================================")

print("\nIncumbent:")
print(best_x, "->", best_y)

print("\nWeek 10:")
print(week10_x, "->", week10_y)

print("\nWeek 11:")
print(week11_x, "->", week11_y)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    1.25 * nearest_distance,
    0.05
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 7. Conservative trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.15 * lengthscales,
    0.015,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(42)

candidates = rng.uniform(
    lower,
    upper,
    size=(450000, 5)
)

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(
            x[j] - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j] - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 15. Axis profiles
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(5):

    values = np.linspace(
        lower[j],
        upper[j],
        30000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(profile_mu)

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 16. Recent-direction line diagnostic
# ------------------------------------------------------------
#
# Direction runs from incumbent toward Week 11.
# Negative t moves in the opposite direction.
#
# Range is automatically limited by the trust-region box.
# ------------------------------------------------------------

direction = (
    week11_x - best_x
)

direction_norm = np.linalg.norm(
    direction
)

unit_direction = (
    direction
    / direction_norm
)

# Find how far we can travel in each direction
# without leaving the trust region.

t_pos_limits = []
t_neg_limits = []

for j in range(5):

    if unit_direction[j] > 0:

        t_pos_limits.append(
            (upper[j] - best_x[j])
            / unit_direction[j]
        )

        t_neg_limits.append(
            (best_x[j] - lower[j])
            / unit_direction[j]
        )

    elif unit_direction[j] < 0:

        t_pos_limits.append(
            (lower[j] - best_x[j])
            / unit_direction[j]
        )

        t_neg_limits.append(
            (best_x[j] - upper[j])
            / unit_direction[j]
        )

t_pos_max = min(t_pos_limits)
t_neg_max = min(t_neg_limits)

t_values = np.linspace(
    -t_neg_max,
    t_pos_max,
    50000
)

line_candidates = (
    best_x
    + t_values[:, None]
    * unit_direction
)

line_distance, _ = tree.query(
    line_candidates,
    k=1
)

line_valid = line_candidates[
    line_distance > 0.01
]

if len(line_valid) > 0:

    line_mu, line_sigma = gp.predict(
        line_valid,
        return_std=True
    )

    line_idx = np.argmax(
        line_mu
    )

    print("\n================================")
    print("RECENT-DIRECTION LINE")
    print("================================")

    print(
        "Best line candidate =",
        line_valid[line_idx]
    )

    print(
        "mean =",
        line_mu[line_idx]
    )

    print(
        "std =",
        line_sigma[line_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            line_valid[line_idx]
            - best_x
        )
    )

else:

    print("\nNo admissible recent-direction points.")

DATA
X shape: (31, 5)
Y shape: (31,)

Current best:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.6688929556550234

WEEK 11 CALIBRATION CHECK
Predicted mean: -0.23128728702383516
Predicted std : 0.03895291141835071
Actual        : -0.20392262491163674

Prediction error:
0.02736466211219843

Error / predicted std:
0.7025062085425261

RECENT LOCAL POINTS

Incumbent:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Week 10:
[0.445661 0.406946 0.588674 0.702373 0.121625] -> -0.2629443056177093

Week 11:
[0.444632 0.410696 0.586583 0.71196  0.118032] -> -0.20392262491163674

GP FIT

Fitted kernel:
1.38**2 * Matern(length_scale=[0.889, 1.03, 1.45, 1.01, 1.15], nu=2.5) + WhiteKernel(noise_level=0.00255)

ARD lengthscales:
[0.88916186 1.0330442  1.45076778 1.01300697 1.1543555 ]

Normalised inverse-lengthscale sensitivity:
[0.24262289 0.20883039 0.14870128 0.21296105 0.18688439]

EMPIR

In [2]:
# ============================================================
# FINAL FUNCTION 6 - WEEK 12 SELECTION
# ============================================================
#
# The unrestricted posterior-mean / low-UCB candidate lies very
# close to the Week 10 / Week 11 region and pushes x4 downward
# again toward the poorer Week 10 configuration.
#
# The recent-direction diagnostic gives a more conservative
# interpolation from Week 11 back toward the proven incumbent.
#
# Week 11 was already close to the best observed value and was
# +0.70 sigma better than its GP prediction, so this controlled
# move is preferred over another larger multidimensional step.
# ============================================================

week12_candidate = np.array([
    0.444319,
    0.410445,
    0.594906,
    0.717081,
    0.120120
])

print("================================")
print("FUNCTION 6 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction
final_mu, final_sigma = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check distance to all previous observations
nearest_dist, nearest_idx = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 6 - WEEK 12 FINAL

Candidate:
[0.444319 0.410445 0.594906 0.717081 0.12012 ]

Predicted mean:
-0.2270048151578774

Predicted std:
0.03622313142167735

Distance from current best:
0.04436743740627803

Nearest observed-point distance:
0.010000884160913007

Nearest observed point:
[0.444632 0.410696 0.586583 0.71196  0.118032]

Portal format:
0.444319-0.410445-0.594906-0.717081-0.120120
